# SOB4ES - Extractor automatico de resultados entre ramas

Este notebook automatiza lo que se ha estado haciendo a mano hasta ahora: leer los notebooks de una rama de git (sin necesidad de hacer `checkout`, usando `git show rama:archivo`), extraer las metricas de evaluacion, el numero de variables activas, y detectar automaticamente dos problemas que ya han aparecido varias veces de forma manual:

1. **Variables que no coinciden** entre lo que dice `FEATURES_AUTORIZADAS` y lo que realmente se cargo en `X_train` (columna comentada pero no re-ejecutado).
2. **Celdas ejecutadas fuera de orden** (`execution_count` no creciente de arriba a abajo), que es la causa raiz de los resultados "pegados" de una ejecucion anterior (el bug que se detecto en `regressorchain.ipynb` y `mlp_custom_loss.ipynb`).

Compara dos ramas cualesquiera (p. ej. `main` contra `prueba1-iteracion-2`) para los 8 notebooks de modelos, y genera la misma tabla comparativa que se ha ido pidiendo manualmente en el chat.

**Requisito:** tener el repositorio clonado localmente (funciona sobre tu copia local de git, sin necesidad de subir nada a ningun sitio). No hace falta hacer `git checkout` de las ramas - se leen los archivos directamente del historial de git.


## 0.- Configuración

In [21]:
import os
import json
import re
import pandas as pd
from IPython.display import display
import subprocess

# --- EDITA ESTO ---
GIT_REPO_PATH = "."  # Usar '.' si ejecutas el notebook dentro del repositorio

# Modifica para definir las ramas a comparar
BRANCHES = [
    "model-prep-var",
    "model-prep-var-1",
    "model-prep-var-2",
    "model-prep-var-1-2",
    "model-prep-var-3",
    "model-prep-var-1-3",
    "model-prep-var-2-3",
    "model-prep-var-1-2-3",
    # "model-prep-var-x-y",   Añade más ramas según necesites
]

# Notebooks a analizar
NOTEBOOKS = [
    "reg-model.ipynb",
    "rf_model.ipynb",
    "rf_multisalida.ipynb",
    "regressorchain.ipynb",
    "xgboost_model.ipynb",
    "xgb_multisalida.ipynb",
    "mlp_multisalida.ipynb",
    "mlp_custom_loss.ipynb",
]

NOTEBOOKS_SUBDIR = ""

# Lista completa de targets a extraer y comparar
TARGETS = [
    # Shannon diversity index
    'nematode_shannon_z',
    'macro_shannon_z',
    'earthworm_shannon_z',
    'orib_shannon_z',
    'meso_shannon_z',
    'coll_shannon_z',
    'bac_shannon_z',
    'fun_shannon_z',
    'euk_shannon_z',
    'oomy_shannon_z',
    'cerc_shannon_z',
    # Richness
    'macro_order_richness_z',
    'earthworm_richness_z',
    'orib_species_richness_z',
    'meso_species_richness_z',
    'coll_species_richness_z',
    'bac_asv_richness_z',
    'fun_asv_richness_z',
    'euk_asv_richness_z',
    'oomy_asv_richness_z',
    'cerc_asv_richness_z',
]

# Define los targets prioritarios que quieres destacar en la tabla comparativa
TARGETS_PRIORITARIOS = ["earthworm_shannon_z", "earthworm_richness_z"]

## 1.- Funciones de lectura desde git

`obtener_notebook_desde_git` usa `git show <rama>:<archivo>` para leer el contenido de un archivo tal y como esta en una rama concreta, sin tocar el working directory ni hacer checkout. Si el repo es remoto y no lo tienes clonado, hay una alternativa comentada al final de la celda usando la API de GitHub (`raw.githubusercontent.com`).

In [22]:
def obtener_notebook_desde_git(repo_path, branch, filename, subdir=""):
    """Lee un .ipynb tal y como está en una rama concreta, vía `git show`, sin checkout."""
    ruta_relativa = os.path.join(subdir, filename) if subdir else filename
    try:
        resultado = subprocess.run(
            ["git", "-C", repo_path, "show", f"{branch}:{ruta_relativa}"],
            capture_output=True,
            text=True,
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print(
            f"  [ERROR] No se pudo leer {filename} en la rama {branch}: {e.stderr.strip()}"
        )
        return None
    return json.loads(resultado.stdout)


def listar_ramas(repo_path):
    """Útil para comprobar el nombre exacto de las ramas disponibles (locales y remotas)."""
    resultado = subprocess.run(
        ["git", "-C", repo_path, "branch", "-a"],
        capture_output=True,
        text=True,
    )
    print(resultado.stdout)

## 2.- Funciones de extracción

- `extraer_features_activas`: parsea la lista `FEATURES_AUTORIZADAS` y separa las lineas comentadas (excluidas) de las activas.
- `extraer_shape_xtrain`: busca el print de `X_train: (filas, columnas)` para saber cuantas variables se usaron realmente en el entrenamiento.
- `detectar_celdas_desordenadas`: compara el `execution_count` de las celdas de codigo en el orden en que aparecen en el notebook; si no es creciente, señala un posible problema de ejecucion (resultados de una corrida anterior, no de la actual).
- `extraer_metricas_targets`: busca en los outputs de texto, **a partir del marcador "Evaluacion final sobre eval.csv"**, las lineas con R2/RMSE/MAE para los targets pedidos. Restringir la busqueda a ese bloque es importante: sin eso, el regex puede confundirse con el R2 de entrenamiento/CV impreso en las celdas de tuning (mucho mas alto por sobreajuste) y dar una lectura falsa. Soporta dos formatos: modelos single-target (R2+RMSE+MAE por target) y modelos multisalida (solo R2 por target, con RMSE/MAE unicamente a nivel global).

In [23]:
def extraer_features_activas(nb_json):
    for c in nb_json["cells"]:
        src = "".join(c.get("source", []))
        if "FEATURES_AUTORIZADAS =" in src:
            m = re.search(r"FEATURES_AUTORIZADAS\s*=\s*\[(.*?)\]", src, re.S)
            if not m:
                continue
            lineas = [l.strip() for l in m.group(1).split("\n") if l.strip()]
            activas, excluidas = [], []
            for linea in lineas:
                nombre_m = re.search(r"'([^']+)'", linea)
                if not nombre_m:
                    continue
                nombre = nombre_m.group(1)
                if linea.startswith("#"):
                    excluidas.append(nombre)
                else:
                    activas.append(nombre)
            return activas, excluidas
    return [], []


def extraer_shape_xtrain(nb_json):
    for c in nb_json["cells"]:
        for out in c.get("outputs", []):
            texto = "".join(out.get("text", []))
            m = re.search(r"X_train:\s*\((\d+),\s*(\d+)\)", texto)
            if m:
                return int(m.group(1)), int(m.group(2))
    return None, None


def detectar_celdas_desordenadas(nb_json):
    """Devuelve una lista de avisos si el execution_count no es creciente."""
    avisos = []
    ultimo_exec = None
    for i, c in enumerate(nb_json["cells"]):
        if c.get("cell_type") != "code":
            continue
        exec_count = c.get("execution_count")
        if exec_count is None:
            continue
        if ultimo_exec is not None and exec_count < ultimo_exec:
            avisos.append(
                f"celda #{i} tiene execution_count={exec_count}, "
                f"menor que una celda anterior ({ultimo_exec}) -> posible resultado obsoleto"
            )
        ultimo_exec = exec_count
    return avisos


MARCADOR_EVAL = "Evaluacion final sobre eval.csv"


def extraer_metricas_targets(nb_json, targets):
    """Extrae R2/RMSE/MAE de la evaluación final para cada target."""
    resultados = {}
    patron_completo = {
        t: re.compile(
            rf"{re.escape(t)}\s+([\-0-9.]+)\s+([\-0-9.]+)\s+([\-0-9.]+)"
        )
        for t in targets
    }
    patron_solo_r2 = {
        t: re.compile(rf"{re.escape(t)}\s+([\-0-9.]+)\s*$", re.M)
        for t in targets
    }
    patron_global = re.compile(
        r"R2\s+global:\s*([\-0-9.]+).*?RMSE\s+global:\s*([\-0-9.]+).*?MAE\s+global:\s*([\-0-9.]+)",
        re.S,
    )

    for c in nb_json["cells"]:
        for out in c.get("outputs", []):
            texto_completo = "".join(out.get("text", []))
            if MARCADOR_EVAL not in texto_completo:
                continue
            texto = texto_completo[texto_completo.index(MARCADOR_EVAL) :]

            g = patron_global.search(texto)
            global_metrics = (
                {
                    "r2_global": float(g.group(1)),
                    "rmse_global": float(g.group(2)),
                    "mae_global": float(g.group(3)),
                }
                if g
                else None
            )

            for t in targets:
                if t in resultados:
                    continue
                m = patron_completo[t].search(texto)
                if m:
                    resultados[t] = {
                        "r2": float(m.group(1)),
                        "rmse": float(m.group(2)),
                        "mae": float(m.group(3)),
                    }
                    continue
                m2 = patron_solo_r2[t].search(texto)
                if m2:
                    resultados[t] = {
                        "r2": float(m2.group(1)),
                        "rmse": None,
                        "mae": None,
                        "global": global_metrics,
                    }
    return resultados

## 3.- Extracción de notebooks y por rama

In [24]:
def analizar_notebook(repo_path, branch, filename, subdir, targets):
    nb_json = obtener_notebook_desde_git(repo_path, branch, filename, subdir)
    if nb_json is None:
        return None

    activas, excluidas = extraer_features_activas(nb_json)
    filas, columnas = extraer_shape_xtrain(nb_json)
    metricas = extraer_metricas_targets(nb_json, targets)
    avisos_orden = detectar_celdas_desordenadas(nb_json)

    return {
        "n_features_lista": len(activas),
        "features_excluidas": excluidas,
        "x_train_shape": (filas, columnas),
        "coherente": (
            (columnas == len(activas)) if columnas is not None else None
        ),
        "metricas": metricas,
        "avisos_orden": avisos_orden,
    }


resultados_por_rama = {}

for branch in BRANCHES:
    print(f"Leyendo notebooks de la rama: {branch}...")
    resultados_por_rama[branch] = {}
    for nb_name in NOTEBOOKS:
        print(f"  {nb_name}")
        resultados_por_rama[branch][nb_name] = analizar_notebook(
            GIT_REPO_PATH,
            branch,
            nb_name,
            NOTEBOOKS_SUBDIR,
            TARGETS,
        )
    print()

Leyendo notebooks de la rama: model-prep-var...
  reg-model.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-1...
  reg-model.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-2...
  reg-model.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-1-2...
  reg-model.ipynb
  rf_model.ipynb
  rf_multisalida.ipynb
  regressorchain.ipynb
  xgboost_model.ipynb
  xgb_multisalida.ipynb
  mlp_multisalida.ipynb
  mlp_custom_loss.ipynb

Leyendo notebooks de la rama: model-prep-var-3...
  reg-model.ipynb
  rf_model.ipynb
  rf_multis

## 4. Validaciones automaticas

Antes de comparar metricas, se comprueba automaticamente lo que hasta ahora se ha ido revisando a mano:
- ¿El numero de columnas de `X_train` coincide con el numero de variables activas en `FEATURES_AUTORIZADAS`?
- ¿Hay celdas con `execution_count` fuera de orden (posible resultado obsoleto)?
- ¿Los valores de los targets prioritarios son sospechosamente identicos entre ambas ramas (posible notebook no re-ejecutado)?

In [25]:
print("=" * 100)
print("VALIDACIONES AUTOMÁTICAS")
print("=" * 100)

base_branch = BRANCHES[0]

for nb_name in NOTEBOOKS:
    print(f"\n[{nb_name}]")

    # 1. Comprobar coherencia y orden por cada rama
    for branch in BRANCHES:
        r = resultados_por_rama.get(branch, {}).get(nb_name)
        if r is None:
            print(f"  [ERROR - {branch}] No se pudo leer el notebook.")
            continue

        filas, columnas = r["x_train_shape"]
        if r["coherente"] is False:
            print(
                f"  [AVISO - {branch}] X_train tiene {columnas} columnas pero "
                f"FEATURES_AUTORIZADAS tiene {r['n_features_lista']} activas -> revisar notebook"
            )
        if r["avisos_orden"]:
            print(f"  [AVISO - {branch}] Celdas ejecutadas fuera de orden:")
            for a in r["avisos_orden"]:
                print(f"      - {a}")

    # 2. Comprobar si hay métricas idénticas respecto a la primera rama (baseline)
    r_base = resultados_por_rama.get(base_branch, {}).get(nb_name)
    if r_base:
        for comp_branch in BRANCHES[1:]:
            r_comp = resultados_por_rama.get(comp_branch, {}).get(nb_name)
            if not r_comp:
                continue
            for t in TARGETS_PRIORITARIOS:
                m_base = r_base["metricas"].get(t)
                m_comp = r_comp["metricas"].get(t)
                if (
                    m_base
                    and m_comp
                    and m_base.get("r2") == m_comp.get("r2")
                ):
                    print(
                        f"  [AVISO] {t}: R2 IDENTICO en '{base_branch}' y '{comp_branch}' ({m_base['r2']}) "
                        f"-> revisar si se re-ejecutó de verdad"
                    )

VALIDACIONES AUTOMÁTICAS

[reg-model.ipynb]
  [AVISO - model-prep-var] Celdas ejecutadas fuera de orden:
      - celda #8 tiene execution_count=5, menor que una celda anterior (7) -> posible resultado obsoleto

[rf_model.ipynb]

[rf_multisalida.ipynb]
  [AVISO - model-prep-var] Celdas ejecutadas fuera de orden:
      - celda #8 tiene execution_count=17, menor que una celda anterior (21) -> posible resultado obsoleto

[regressorchain.ipynb]

[xgboost_model.ipynb]

[xgb_multisalida.ipynb]

[mlp_multisalida.ipynb]

[mlp_custom_loss.ipynb]


## 5- Tablas comparativas

En esta sección se generarán múltiples tablas para facilitar la comparación de resultados.

In [26]:
filas_tabla = []
base_branch = BRANCHES[0]

for nb_name in NOTEBOOKS:
    fila = {"Notebook": nb_name}

    for t in TARGETS_PRIORITARIOS:
        # Obtener R2 de la rama base
        r_base = resultados_por_rama.get(base_branch, {}).get(nb_name)
        m_base = r_base["metricas"].get(t) if r_base else None
        r2_base = m_base["r2"] if m_base else None

        fila[f"{t}_R2 ({base_branch})"] = r2_base

        # R2 y Delta para las ramas a comparar
        for branch in BRANCHES[1:]:
            r_branch = resultados_por_rama.get(branch, {}).get(nb_name)
            m_branch = r_branch["metricas"].get(t) if r_branch else None
            r2_val = m_branch["r2"] if m_branch else None

            fila[f"{t}_R2 ({branch})"] = r2_val

            delta = (
                round(r2_val - r2_base, 4)
                if (r2_val is not None and r2_base is not None)
                else None
            )
            fila[f"{t}_delta ({branch})"] = delta

    filas_tabla.append(fila)

df_comparativa = pd.DataFrame(filas_tabla)

print(f"Comparación entre {len(BRANCHES)} ramas (Base: {base_branch}):\n")
print(df_comparativa.to_string(index=False))

# Guardar resultado en CSV
os.makedirs("output/comparativas", exist_ok=True)
nombre_salida = f"comparativa_{'_vs_'.join(BRANCHES)}.csv".replace("/", "-")
ruta_csv = f"output/comparativas/{nombre_salida}"
df_comparativa.to_csv(ruta_csv, index=False)
print(f"\nGuardado en: {ruta_csv}")

Comparación entre 8 ramas (Base: model-prep-var):

             Notebook  earthworm_shannon_z_R2 (model-prep-var)  earthworm_shannon_z_R2 (model-prep-var-1)  earthworm_shannon_z_delta (model-prep-var-1)  earthworm_shannon_z_R2 (model-prep-var-2)  earthworm_shannon_z_delta (model-prep-var-2)  earthworm_shannon_z_R2 (model-prep-var-1-2)  earthworm_shannon_z_delta (model-prep-var-1-2)  earthworm_shannon_z_R2 (model-prep-var-3)  earthworm_shannon_z_delta (model-prep-var-3)  earthworm_shannon_z_R2 (model-prep-var-1-3)  earthworm_shannon_z_delta (model-prep-var-1-3)  earthworm_shannon_z_R2 (model-prep-var-2-3)  earthworm_shannon_z_delta (model-prep-var-2-3)  earthworm_shannon_z_R2 (model-prep-var-1-2-3)  earthworm_shannon_z_delta (model-prep-var-1-2-3)  earthworm_richness_z_R2 (model-prep-var)  earthworm_richness_z_R2 (model-prep-var-1)  earthworm_richness_z_delta (model-prep-var-1)  earthworm_richness_z_R2 (model-prep-var-2)  earthworm_richness_z_delta (model-prep-var-2)  earthworm_richness

### 5.1.- Tabla de compraración general

In [27]:
filas_general = []
base_branch = BRANCHES[0]

for nb_name in NOTEBOOKS:
    fila = {"Notebook": nb_name}

    for branch in BRANCHES:
        res = resultados_por_rama.get(branch, {}).get(nb_name)

        if res:
            # Dimensiones de X_train
            shape_str = (
                f"{res['x_train_shape'][0]}x{res['x_train_shape'][1]}"
                if res["x_train_shape"][0]
                else "N/A"
            )
            fila[f"N_Vars ({branch})"] = res["n_features_lista"]
            fila[f"Shape ({branch})"] = shape_str

            # Extracción de métricas globales
            m_dict = res.get("metricas", {})
            r2_glob, rmse_glob, mae_glob = None, None, None

            # 1. Intentar obtener 'global' si el notebook es multisalida
            for t_info in m_dict.values():
                if t_info and t_info.get("global"):
                    r2_glob = t_info["global"].get("r2_global")
                    rmse_glob = t_info["global"].get("rmse_global")
                    mae_glob = t_info["global"].get("mae_global")
                    break

            # 2. Si es single-target, promediar el R2 de los targets evaluados
            if r2_glob is None and m_dict:
                r2_vals = [
                    v["r2"]
                    for v in m_dict.values()
                    if v and v.get("r2") is not None
                ]
                if r2_vals:
                    r2_glob = round(sum(r2_vals) / len(r2_vals), 4)

            fila[f"R2_Global ({branch})"] = r2_glob
            if rmse_glob is not None:
                fila[f"RMSE_Global ({branch})"] = rmse_glob
            if mae_glob is not None:
                fila[f"MAE_Global ({branch})"] = mae_glob

            # Calcular Delta R2 Global respecto al baseline
            if branch != base_branch:
                r2_base = fila.get(f"R2_Global ({base_branch})")
                fila[f"Delta_R2_Global ({branch})"] = (
                    round(r2_glob - r2_base, 4)
                    if (r2_glob is not None and r2_base is not None)
                    else None
                )
        else:
            fila[f"N_Vars ({branch})"] = "Error"
            fila[f"R2_Global ({branch})"] = None

    filas_general.append(fila)

df_general = pd.DataFrame(filas_general)

print("=" * 100)
print(f"1. TABLA COMPARATIVA GENERAL (Base: {base_branch})")
print("=" * 100)
print(df_general.to_markdown(index=False))

# Guardar CSV
os.makedirs("output/comparativas", exist_ok=True)
ruta_csv_gen = f"output/comparativas/general_{'_vs_'.join(BRANCHES)}.csv"
df_general.to_csv(ruta_csv_gen, index=False)
print(f"\nGuardado en: {ruta_csv_gen}")

1. TABLA COMPARATIVA GENERAL (Base: model-prep-var)
| Notebook              |   N_Vars (model-prep-var) | Shape (model-prep-var)   |   R2_Global (model-prep-var) |   N_Vars (model-prep-var-1) | Shape (model-prep-var-1)   |   R2_Global (model-prep-var-1) |   Delta_R2_Global (model-prep-var-1) |   N_Vars (model-prep-var-2) | Shape (model-prep-var-2)   |   R2_Global (model-prep-var-2) |   Delta_R2_Global (model-prep-var-2) |   N_Vars (model-prep-var-1-2) | Shape (model-prep-var-1-2)   |   R2_Global (model-prep-var-1-2) |   Delta_R2_Global (model-prep-var-1-2) |   N_Vars (model-prep-var-3) | Shape (model-prep-var-3)   |   R2_Global (model-prep-var-3) |   Delta_R2_Global (model-prep-var-3) |   N_Vars (model-prep-var-1-3) | Shape (model-prep-var-1-3)   |   R2_Global (model-prep-var-1-3) |   Delta_R2_Global (model-prep-var-1-3) |   N_Vars (model-prep-var-2-3) | Shape (model-prep-var-2-3)   |   R2_Global (model-prep-var-2-3) |   Delta_R2_Global (model-prep-var-2-3) |   N_Vars (model-prep-var-1

### 5.2.- Comparación por targets

In [28]:
from collections import Counter
from IPython.display import display, HTML

base_branch = BRANCHES[0]
tablas_por_target = {}
html_partes = []

def obtener_label_rama(branch):
    """Etiqueta la rama con la(s) variable(s) que excluye, usando el conjunto de
    excluidas más frecuente entre notebooks (por si algún notebook individual
    está desincronizado). Si no excluye nada, se etiqueta como 'Baseline'."""
    conteos = Counter()
    for nb_name in NOTEBOOKS:
        res = resultados_por_rama.get(branch, {}).get(nb_name)
        if res:
            conteos[tuple(sorted(res["features_excluidas"]))] += 1
    if not conteos:
        return branch
    excluidas_comunes = conteos.most_common(1)[0][0]
    return "+".join(excluidas_comunes) if excluidas_comunes else "Baseline"

LABELS_RAMA = {branch: obtener_label_rama(branch) for branch in BRANCHES}

os.makedirs("output/comparativas", exist_ok=True)

for target in TARGETS:
    filas_target = []

    for nb_name in NOTEBOOKS:
        fila = {"Notebook": nb_name}

        for branch in BRANCHES:
            res = resultados_por_rama.get(branch, {}).get(nb_name)
            m = (
                res["metricas"].get(target)
                if (res and res.get("metricas"))
                else None
            )
            fila[f"R2 ({LABELS_RAMA[branch]})"] = m["r2"] if m else None

        filas_target.append(fila)

    df_target = pd.DataFrame(filas_target)

    # Fila final con la media global (de todos los notebooks) por rama
    cols_r2 = [c for c in df_target.columns if c != "Notebook"]
    fila_media = {"Notebook": "MEDIA GLOBAL"}
    for col in cols_r2:
        fila_media[col] = df_target[col].mean(skipna=True)
    df_target = pd.concat([df_target, pd.DataFrame([fila_media])], ignore_index=True)

    tablas_por_target[target] = df_target

    html_target = df_target.to_html(
        index=False, na_rep="—", float_format=lambda x: f"{x:.4f}"
    )

    print("=" * 100)
    print(f"2. TABLA COMPARATIVA POR TARGET: {target} (Base: {LABELS_RAMA[base_branch]})")
    print("=" * 100)
    display(HTML(html_target))

    # Guardar HTML (uno por target)
    ruta_html_target = f"output/comparativas/por_target_{target}_{'_vs_'.join(BRANCHES)}.html"
    with open(ruta_html_target, "w", encoding="utf-8") as f:
        f.write(html_target)
    print(f"Guardado en: {ruta_html_target}\n")

    html_partes.append(f"<h2>{target}</h2>\n{html_target}")

# --- Export final con todas las tablas juntas ---
html_final = (
    "<html><head><meta charset='utf-8'>"
    "<style>table{border-collapse:collapse;margin-bottom:30px;} "
    "th,td{border:1px solid #ccc;padding:4px 8px;text-align:right;} "
    "th{background:#f0f0f0;} td:first-child,th:first-child{text-align:left;}</style>"
    "</head><body>"
    f"<h1>Comparativa por targets (Base: {LABELS_RAMA[base_branch]})</h1>"
    + "\n".join(html_partes)
    + "</body></html>"
)

ruta_html_final = f"output/comparativas/todas_las_tablas_{'_vs_'.join(BRANCHES)}.html"
with open(ruta_html_final, "w", encoding="utf-8") as f:
    f.write(html_final)

print("=" * 100)
print(f"Export final con todas las tablas guardado en: {ruta_html_final}")
print("=" * 100)

2. TABLA COMPARATIVA POR TARGET: nematode_shannon_z (Base: Baseline)


Notebook,R2 (Baseline),R2 (cu_z),R2 (ni_z),R2 (cu_z+ni_z),R2 (mo_z),R2 (cu_z+mo_z),R2 (mo_z+ni_z),R2 (cu_z+mo_z+ni_z)
reg-model.ipynb,0.0112,0.0063,0.0031,-0.0019,0.0073,0.0073,0.0073,0.0073
rf_model.ipynb,0.1366,0.1613,0.1493,0.1350,0.1607,0.1437,0.1406,0.1360
rf_multisalida.ipynb,0.1545,0.1457,0.1429,0.1514,0.1493,0.1645,0.1541,0.1527
regressorchain.ipynb,0.1525,0.1578,0.1483,0.1489,0.1568,0.1551,0.1525,0.1434
xgboost_model.ipynb,0.1637,0.1679,0.1622,0.1461,0.1692,0.1600,0.1447,0.1492
xgb_multisalida.ipynb,0.2228,0.2193,0.2110,0.2163,0.2245,0.2339,0.2285,0.2189
mlp_multisalida.ipynb,0.0829,0.0697,0.0704,0.0719,0.0756,0.0526,0.0665,0.0656
mlp_custom_loss.ipynb,0.0668,0.0699,0.0579,0.0584,0.0667,0.1078,0.0575,0.0482
MEDIA GLOBAL,0.1239,0.1247,0.1181,0.1158,0.1263,0.1281,0.1190,0.1152


Guardado en: output/comparativas/por_target_nematode_shannon_z_model-prep-var_vs_model-prep-var-1_vs_model-prep-var-2_vs_model-prep-var-1-2_vs_model-prep-var-3_vs_model-prep-var-1-3_vs_model-prep-var-2-3_vs_model-prep-var-1-2-3.html

2. TABLA COMPARATIVA POR TARGET: macro_shannon_z (Base: Baseline)


Notebook,R2 (Baseline),R2 (cu_z),R2 (ni_z),R2 (cu_z+ni_z),R2 (mo_z),R2 (cu_z+mo_z),R2 (mo_z+ni_z),R2 (cu_z+mo_z+ni_z)
reg-model.ipynb,0.1257,0.1174,0.1282,0.1191,0.1341,0.1341,0.1341,0.1341
rf_model.ipynb,0.2797,0.2619,0.2735,0.2694,0.2777,0.2740,0.2851,0.2788
rf_multisalida.ipynb,0.2000,0.1892,0.1894,0.1966,0.1936,0.2026,0.2036,0.2103
regressorchain.ipynb,0.3001,0.2915,0.2899,0.2906,0.2930,0.2913,0.3016,0.2995
xgboost_model.ipynb,0.2601,0.2643,0.2668,0.2582,0.2668,0.2578,0.2598,0.2697
xgb_multisalida.ipynb,0.3771,0.3650,0.3677,0.3795,0.3825,0.3958,0.3836,0.3894
mlp_multisalida.ipynb,0.2396,0.2286,0.2202,0.2027,0.2163,0.2153,0.2324,0.2066
mlp_custom_loss.ipynb,0.2704,0.2461,0.2303,0.2098,0.2300,0.2908,0.2326,0.1918
MEDIA GLOBAL,0.2566,0.2455,0.2457,0.2407,0.2493,0.2577,0.2541,0.2475


Guardado en: output/comparativas/por_target_macro_shannon_z_model-prep-var_vs_model-prep-var-1_vs_model-prep-var-2_vs_model-prep-var-1-2_vs_model-prep-var-3_vs_model-prep-var-1-3_vs_model-prep-var-2-3_vs_model-prep-var-1-2-3.html

2. TABLA COMPARATIVA POR TARGET: earthworm_shannon_z (Base: Baseline)


...

Guardado en: output/comparativas/por_target_euk_asv_richness_z_model-prep-var_vs_model-prep-var-1_vs_model-prep-var-2_vs_model-prep-var-1-2_vs_model-prep-var-3_vs_model-prep-var-1-3_vs_model-prep-var-2-3_vs_model-prep-var-1-2-3.html

2. TABLA COMPARATIVA POR TARGET: oomy_asv_richness_z (Base: Baseline)


Notebook,R2 (Baseline),R2 (cu_z),R2 (ni_z),R2 (cu_z+ni_z),R2 (mo_z),R2 (cu_z+mo_z),R2 (mo_z+ni_z),R2 (cu_z+mo_z+ni_z)
reg-model.ipynb,0.1035,0.1124,0.1038,0.1124,0.0987,0.0987,0.0987,0.0987
rf_model.ipynb,0.1895,0.1882,0.1925,0.1957,0.1952,0.2007,0.2007,0.2018
rf_multisalida.ipynb,0.1917,0.1825,0.1845,0.1992,0.1829,0.2006,0.1986,0.1998
regressorchain.ipynb,0.2135,0.2194,0.2191,0.2186,0.2288,0.2296,0.2319,0.2288
xgboost_model.ipynb,0.1776,0.1854,0.1884,0.1859,0.1946,0.1910,0.1949,0.1917
xgb_multisalida.ipynb,0.1650,0.1794,0.1753,0.1728,0.1805,0.1788,0.1859,0.1875
mlp_multisalida.ipynb,0.0640,0.0944,0.0980,0.0961,0.1171,0.1027,0.0540,0.1102
mlp_custom_loss.ipynb,0.0188,0.0598,0.0513,0.0717,0.0633,0.0530,0.0706,0.0905
MEDIA GLOBAL,0.1404,0.1527,0.1516,0.1565,0.1576,0.1569,0.1544,0.1636


Guardado en: output/comparativas/por_target_oomy_asv_richness_z_model-prep-var_vs_model-prep-var-1_vs_model-prep-var-2_vs_model-prep-var-1-2_vs_model-prep-var-3_vs_model-prep-var-1-3_vs_model-prep-var-2-3_vs_model-prep-var-1-2-3.html

2. TABLA COMPARATIVA POR TARGET: cerc_asv_richness_z (Base: Baseline)


Notebook,R2 (Baseline),R2 (cu_z),R2 (ni_z),R2 (cu_z+ni_z),R2 (mo_z),R2 (cu_z+mo_z),R2 (mo_z+ni_z),R2 (cu_z+mo_z+ni_z)
reg-model.ipynb,0.1019,0.0935,0.0985,0.0890,0.0941,0.0941,0.0941,0.0941
rf_model.ipynb,0.1207,0.1274,0.1254,0.1211,0.1224,0.1215,0.1160,0.1257
rf_multisalida.ipynb,0.1171,0.1149,0.1128,0.1217,0.1124,0.1157,0.1169,0.1210
regressorchain.ipynb,0.1147,0.1183,0.1108,0.1175,0.1084,0.1174,0.1089,0.1157
xgboost_model.ipynb,0.1227,0.1351,0.1248,0.1364,0.1172,0.1311,0.1143,0.1205
xgb_multisalida.ipynb,0.0983,0.1027,0.1022,0.1028,0.0949,0.0743,0.0831,0.0854
mlp_multisalida.ipynb,0.0285,0.0547,0.0548,0.0627,0.0596,0.0503,0.0458,0.0662
mlp_custom_loss.ipynb,-0.0845,0.0474,0.0338,0.0193,0.0546,-0.0407,0.0189,0.0048
MEDIA GLOBAL,0.0774,0.0993,0.0954,0.0963,0.0955,0.0830,0.0872,0.0917


Guardado en: output/comparativas/por_target_cerc_asv_richness_z_model-prep-var_vs_model-prep-var-1_vs_model-prep-var-2_vs_model-prep-var-1-2_vs_model-prep-var-3_vs_model-prep-var-1-3_vs_model-prep-var-2-3_vs_model-prep-var-1-2-3.html

Export final con todas las tablas guardado en: output/comparativas/todas_las_tablas_model-prep-var_vs_model-prep-var-1_vs_model-prep-var-2_vs_model-prep-var-1-2_vs_model-prep-var-3_vs_model-prep-var-1-3_vs_model-prep-var-2-3_vs_model-prep-var-1-2-3.html


## 6. Variables excluidas por rama

Muestra que variables estan comentadas (excluidas) en `FEATURES_AUTORIZADAS` en cada rama, para confirmar rapidamente que la rama de comparacion excluye la variable esperada (y solo esa) en los 8 notebooks.

In [29]:
for nb_name in NOTEBOOKS:
    print(f"\n--- {nb_name} ---")
    for branch in BRANCHES:
        res = resultados_por_rama.get(branch, {}).get(nb_name)
        if res:
            excluidas = res["features_excluidas"]
            print(f"  [{branch:25}] Excluidas ({len(excluidas)}): {excluidas}")
        else:
            print(f"  [{branch:25}] Error al leer notebook")


--- reg-model.ipynb ---
  [model-prep-var           ] Excluidas (0): []
  [model-prep-var-1         ] Excluidas (1): ['cu_z']
  [model-prep-var-2         ] Excluidas (1): ['ni_z']
  [model-prep-var-1-2       ] Excluidas (2): ['cu_z', 'ni_z']
  [model-prep-var-3         ] Excluidas (1): ['mo_z']
  [model-prep-var-1-3       ] Excluidas (1): ['mo_z']
  [model-prep-var-2-3       ] Excluidas (1): ['mo_z']
  [model-prep-var-1-2-3     ] Excluidas (1): ['mo_z']

--- rf_model.ipynb ---
  [model-prep-var           ] Excluidas (0): []
  [model-prep-var-1         ] Excluidas (1): ['cu_z']
  [model-prep-var-2         ] Excluidas (1): ['ni_z']
  [model-prep-var-1-2       ] Excluidas (2): ['cu_z', 'ni_z']
  [model-prep-var-3         ] Excluidas (1): ['mo_z']
  [model-prep-var-1-3       ] Excluidas (2): ['cu_z', 'mo_z']
  [model-prep-var-2-3       ] Excluidas (2): ['mo_z', 'ni_z']
  [model-prep-var-1-2-3     ] Excluidas (3): ['cu_z', 'mo_z', 'ni_z']

--- rf_multisalida.ipynb ---
  [model-prep-var    